In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

In [ ]:
df = pd.read_csv("data/transit_ridership.csv")

In [2]:
print(df.head())

         date route_id direction  boarding_count  alighting_count  \
0  2024-10-21     R205   Inbound           388.0             57.0   
1  2024-08-29     R105  Outbound           457.0            169.0   
2  2024-08-31     R207   Inbound             NaN            518.0   
3  2024-12-24     R208  Outbound           354.0            302.0   
4  2024-03-05     R207   Inbound           423.0            643.0   

      vehicle_type  trip_duration_min   weather  temperature_c is_holiday  
0          Minibus               77.9     Clear          -14.3      false  
1  Articulated Bus               76.6      Snow           -1.1      false  
2  Articulated Bus               58.5  Overcast           22.8      False  
3  Articulated Bus               47.4      Snow           29.6          0  
4          Minibus               17.4  Overcast          -11.7         No  


In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1993 entries, 0 to 1992
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               1993 non-null   str    
 1   route_id           1993 non-null   str    
 2   direction          1993 non-null   str    
 3   boarding_count     1854 non-null   float64
 4   alighting_count    1907 non-null   float64
 5   vehicle_type       1993 non-null   str    
 6   trip_duration_min  1883 non-null   float64
 7   weather            1993 non-null   str    
 8   temperature_c      1940 non-null   float64
 9   is_holiday         1993 non-null   str    
dtypes: float64(4), str(6)
memory usage: 155.8 KB
None


In [4]:
print(df.describe())

       boarding_count  alighting_count  trip_duration_min  temperature_c
count     1854.000000      1907.000000        1883.000000    1940.000000
mean       410.625674       381.549554          56.235582      14.030052
std        222.595905       214.463238          58.949897      16.434646
min         20.000000        15.000000         -12.300000     -15.000000
25%        216.250000       191.500000          32.900000      -0.300000
50%        412.500000       379.000000          52.500000      14.200000
75%        600.000000       565.500000          71.400000      28.500000
max        800.000000       750.000000         999.000000      42.000000


In [5]:
print(df.isnull().sum())

date                   0
route_id               0
direction              0
boarding_count       139
alighting_count       86
vehicle_type           0
trip_duration_min    110
weather                0
temperature_c         53
is_holiday             0
dtype: int64


In [6]:
df = df.copy()

In [7]:
print(df.drop_duplicates())

            date route_id direction  boarding_count  alighting_count  \
0     2024-10-21     R205   Inbound           388.0             57.0   
1     2024-08-29     R105  Outbound           457.0            169.0   
2     2024-08-31     R207   Inbound             NaN            518.0   
3     2024-12-24     R208  Outbound           354.0            302.0   
4     2024-03-05     R207   Inbound           423.0            643.0   
...          ...      ...       ...             ...              ...   
1988  2024-07-07     R104   Inbound           216.0            599.0   
1989  2024-06-11     R104  Outbound            41.0            464.0   
1990  2024-09-29     R205  Outbound           103.0            413.0   
1991  2024-10-21     R206   Inbound           139.0            623.0   
1992  2024-11-12     R302  Outbound           745.0            677.0   

         vehicle_type  trip_duration_min   weather  temperature_c is_holiday  
0             Minibus               77.9     Clear      

In [8]:
df['boarding_count'] = df['boarding_count'].fillna(df['boarding_count'].median())
df['alighting_count'] = df['alighting_count'].fillna(df['alighting_count'].median())
df['trip_duration_min'] = df['trip_duration_min'].fillna(df['trip_duration_min'].median())
df['temperature_c'] = df['temperature_c'].fillna(df['temperature_c'].median())

In [9]:
df['date'] = df['date'].astype(str)

date1 = pd.to_datetime(df['date'], errors='coerce')

date2 = pd.to_datetime(df['date'], errors='coerce', format='%m/%d/%Y')

df['date'] = date1.fillna(date2)

print("Invalid dates:", df['date'].isna().sum())

df = df.dropna(subset=['date'])
print("Cleaned:", df.shape)

Invalid dates: 59
Cleaned: (1934, 10)


In [17]:
df = df[df['route_id'] != 'R999']

In [21]:
holiday_map = {
    'False': False, 'No': False, '0': False, 'false': False,
    'True': True, 'Yes': True, '1': True, 'true':True
}
df['is_holiday'] = df['is_holiday'].map(holiday_map)

In [27]:
df['direction'] = df['direction'].str.lower().str.strip()

df['direction'] = df['direction'].replace({
    'in' : 'inbound',
    'inbnd' : 'inbound',
    'outbnd' : 'outbound',
    'out' : 'outbound' 
})

In [35]:
def clean_vehicle_type(val):
    val = str(val).lower().strip()
    
    if 'mini' in val:
        return 'Minibus'
    elif 'standard' in val or 'std' in val:
        return 'Standard Bus'
    elif 'articulated' in val:
        return 'Articulated Bus'
    else:
        return val

df['vehicle_type'] = df['vehicle_type'].apply(clean_vehicle_type)

In [ ]:
df.loc[df['trip_duration_min'] < 0, 'trip_duration_min'] = np.nan

df['trip_duration_min'] = df['trip_duration_min'].fillna(
    df['trip_duration_min'].median
)

In [38]:
df['month'] = df['date'].dt.to_period('M')
monthly = df.groupby('month')['boarding_count'].sum().sort_index()

In [42]:
summary = {}

# total trips
summary['total_trips'] = len(df)

# date range
summary['date_range'] = f"{df['date'].min().date()} to {df['date'].max().date()}"

# busiest route
route_sum = df.groupby('route_id')['boarding_count'].sum()
summary['busiest_route'] = route_sum.idxmax()

# avg daily ridership
daily = df.groupby('date')['boarding_count'].sum()
summary['avg_daily_ridership'] = round(daily.mean(), 1)

# ridership by vehicle type
summary['ridership_by_vehicle_type'] = df.groupby('vehicle_type')['boarding_count'].sum().to_dict()

# ridership by weather
summary['ridership_by_weather'] = df.groupby('weather')['boarding_count'].sum().to_dict()

# top 5 routes
top_routes = route_sum.sort_values(ascending=False).head(5)

summary['top_5_routes_by_boarding'] = [
    {"route": route, "total_boardings": int(value)}
    for route, value in top_routes.items()
]

summary

{'total_trips': 1931,
 'date_range': '2024-01-01 to 2024-12-31',
 'busiest_route': 'R302',
 'avg_daily_ridership': np.float64(2178.9),
 'ridership_by_vehicle_type': {'Articulated Bus': 255375.5,
  'Minibus': 272736.5,
  'Standard Bus': 265007.5},
 'ridership_by_weather': {'Clear': 196403.0,
  'Overcast': 205571.5,
  'Rain': 205523.5,
  'Snow': 185621.5},
 'top_5_routes_by_boarding': [{'route': 'R302', 'total_boardings': 77840},
  {'route': 'R206', 'total_boardings': 77740},
  {'route': 'R105', 'total_boardings': 77410},
  {'route': 'R208', 'total_boardings': 75190},
  {'route': 'R205', 'total_boardings': 73985}]}

In [ ]:
os.makedirs("output", exist_ok=True)

with open("output/summary.json", "w") as f:
    json.dump(summary, f, indent=4)

In [ ]:
plt.figure()
monthly.plot(marker='o')

plt.title("Monthly Ridership (Total Boardings)")
plt.xlabel("Month")
plt.ylabel("Total Boardings")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("output/monthly_ridership.png")
plt.close()

In [47]:
top_routes.plot(kind='barh')

plt.title("Top Routes by Boardings")
plt.xlabel("Total Boardings")

plt.tight_layout()
plt.savefig("output/top_routes.png")
plt.close()

In [ ]:
df['trip_duration_min'] = pd.to_numeric(df['trip_duration_min'], errors='coerce')


In [51]:
avg_duration = df.groupby('vehicle_type')['trip_duration_min'].mean()

avg_duration.plot(kind='bar')

plt.title("Average Trip Duration by Vehicle Type")
plt.ylabel("Minutes")

plt.tight_layout()
plt.savefig("output/avg_duration_vehicle.png")
plt.close()

In [52]:
avg_boardings_weather = df.groupby('weather')['boarding_count'].mean()

avg_boardings_weather.plot(kind='bar')

plt.title("Average Boardings per Trip by Weather")
plt.ylabel("Boardings")

plt.tight_layout()
plt.savefig("output/avg_boardings_weather.png")
plt.close()